In [ ]:
# =============================================================================
# jobs/27_fw_timing_validation.py
# Was each flood warning issued in time? For every gauge with thresholds, compare when the readings
# crossed the RES level with when the flood warning for its area was issued, against the target lead time.
#
# RUNS IN THE RESTRICTED DEPLOYMENT. It reads the linked thresholds (restricted schema) and writes its table there.
# It reads the readings and the warning periods from the unrestricted schema. Nothing restricted is written to the
# unrestricted schema. Cadence: manual. Run 28 (threshold links) first.
#
# THE IDEA. RES (Type RES, Purpose FW in the thresholds file) is the level at which impacts are expected. A flood warning must be
# out before the reading first crosses it, by at least the target lead time (TargetLeadTimeMinutes, converted to hours).
# Outcomes (rules in utils/fw_timing.py):
#   on time / too short / late / missed        for each upward crossing of RES
#   RES not reached / no readings                for a warning with no crossing while it was in force
# plus trigger_time and issue_delay_hours: when the trigger level (Type ACTCON, Purpose FW) was crossed and how long the
# warning took after it.
#
# THE LINKS (all made by job 28, not redone here). A threshold counts only if it linked to a gauge measure whose unit agrees with
# its own (link_status linked or linked_chosen), so the RES value and the readings are in the same unit. Gauge -> warning: the RES
# row's TargetAreaID against the ea_area_code of the flood-warning-level periods in ea_warning_periods.
#
# WIDGETS
#   res_type, trigger_type   the Type values for the RES level and the trigger level (default RES and ACTCON)
#   purpose                  the Purpose kept (default FW: flood warning; flood alert rows are not validated)
#   season                   the Season kept; blank Season rows are always kept (default All)
#   from_date                only readings and warnings from this date (default 2023-01-01)
#   reset_hours              readings must stay below a level this long before it can be crossed again (default 6)
#   late_hours               a warning issued this long after a crossing is "late", not "missed" (default 24)
#   lookback_hours           how far back to look for the trigger crossing (default 72)
#
# OUTPUT: config.TBL_FW_TIMING (restricted schema), one row per RES crossing or unmatched warning; overwritten in full.
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

import config
from utils.helpers import get_spark, utc_now, table_exists
from utils.fw_timing import gauges_from_links, validate_gauge, warning_issues, summarise, RESULT_COLUMNS

spark = get_spark()

dbutils.widgets.text("res_type", "RES", "Type value for the RES level")
dbutils.widgets.text("trigger_type", "ACTCON", "Type value for the trigger level")
dbutils.widgets.text("purpose", "FW", "Purpose to keep")
dbutils.widgets.text("season", "All", "Season to keep")
dbutils.widgets.text("from_date", "2023-01-01", "From date")
dbutils.widgets.text("reset_hours", "6", "Reset hours")
dbutils.widgets.text("late_hours", "24", "Late hours")
dbutils.widgets.text("lookback_hours", "72", "Trigger look-back hours")

res_type = dbutils.widgets.get("res_type").strip() or "RES"
trigger_type = dbutils.widgets.get("trigger_type").strip() or "ACTCON"
purpose = dbutils.widgets.get("purpose").strip() or "FW"
season = dbutils.widgets.get("season").strip() or "All"
from_date = pd.Timestamp(dbutils.widgets.get("from_date").strip() or "2023-01-01")
reset_hours, late_hours, lookback_hours = (float(dbutils.widgets.get(w) or d) for w, d in (("reset_hours", 6), ("late_hours", 24), ("lookback_hours", 72)))

for t in (config.TBL_FW_THRESHOLD_LINKS, config.TBL_EA_RT_ALL, config.TBL_WARNING_PERIODS, config.TBL_WARNING_EPISODES):
    if not table_exists(spark, t):
        raise Exception(f"{t} does not exist. Run the job that builds it first (28 for the threshold links).")

In [ ]:
# =============================================================================
# GAUGES: the linked RES and trigger levels, one row per gauge, measure and warning area
# =============================================================================

links = spark.table(config.TBL_FW_THRESHOLD_LINKS).toPandas()
print(f"Threshold links: {len(links):,} rows. Status: {links['link_status'].value_counts().to_dict()}")
gauges = gauges_from_links(links, res_type, trigger_type, purpose, season)
print(f"Gauges with a linked {res_type} level (Purpose {purpose}, Season {season}): {len(gauges):,}   "
      f"with a trigger level: {int(gauges['fw_value'].notna().sum()):,}   with a target lead time: {int(gauges['target_lead_hours'].notna().sum()):,}   "
      f"with a warning area: {int(gauges['area_code'].notna().sum()):,}")
if gauges.empty:
    raise Exception("No gauge has a linked RES level. Check job 28's link status counts above, and the res_type, purpose and season widgets.")
gauges["area_key"] = gauges["area_code"].map(lambda v: None if v is None else str(v).strip().upper())

In [ ]:
# =============================================================================
# READINGS AND WARNINGS, THEN THE VALIDATION
# =============================================================================

uris = pd.DataFrame({"measure_uri": gauges["measure_uri"].unique()})
spark.createDataFrame(uris).createOrReplaceTempView("fw_timing_measures")
readings = spark.sql(f"""
    SELECT r.measure_uri, r.date_time, r.value
    FROM {config.TBL_EA_RT_ALL} r JOIN fw_timing_measures m ON m.measure_uri = r.measure_uri
    WHERE r.date_time >= '{from_date.date()}'
""").toPandas()
readings["date_time"] = pd.to_datetime(readings["date_time"]).dt.tz_localize(None) if getattr(pd.to_datetime(readings["date_time"]).dt, "tz", None) else pd.to_datetime(readings["date_time"])
print(f"Readings since {from_date.date()}: {len(readings):,} for {readings['measure_uri'].nunique():,} measures")

periods = spark.table(config.TBL_WARNING_PERIODS).toPandas()
episodes = spark.table(config.TBL_WARNING_EPISODES).toPandas()
issues = warning_issues(periods, episodes, as_of=utc_now().replace(tzinfo=None))
issues = issues[issues["issue_time"] >= from_date].copy()
issues["area_key"] = issues["ea_area_code"].map(lambda v: None if v is None else str(v).strip().upper())
print(f"Gauge warning areas that appear among those episodes: {gauges['area_key'].isin(issues['area_key']).sum():,} of {len(gauges):,} gauges")
print(f"Flood-warning-level episodes since {from_date.date()}: {len(issues):,}")

by_measure = {u: g[["date_time", "value"]] for u, g in readings.groupby("measure_uri")}
rows = []
for gauge in gauges.to_dict("records"):
    series = by_measure.get(gauge["measure_uri"])
    if series is None or gauge["area_key"] is None:
        continue
    area_issues = issues[issues["area_key"] == gauge["area_key"]]
    for r in validate_gauge(gauge, series, area_issues, reset_hours, late_hours, lookback_hours):
        r["measure_uri"] = gauge["measure_uri"]
        rows.append(r)
results = pd.DataFrame(rows, columns=RESULT_COLUMNS + ["measure_uri"])
print(f"Result rows: {len(results):,}")
summary = summarise(results)
print(summary.to_string(index=False))
display(summary)


In [ ]:
# =============================================================================
# WRITE: restricted schema, overwritten in full
# =============================================================================

SCHEMA = StructType([StructField(c, DoubleType() if c in ("lead_hours", "target_lead_hours", "issue_delay_hours", "res_value", "fw_value") else StringType(), True)
                     for c in RESULT_COLUMNS + ["measure_uri"]] + [StructField("computed_at", StringType(), True)])
out = results.copy()
for c in ("res_time", "issue_time", "trigger_time"):
    out[c] = pd.to_datetime(out[c]).dt.strftime("%Y-%m-%dT%H:%M:%S")
out["computed_at"] = utc_now().isoformat()
cols = [f.name for f in SCHEMA.fields]
out = out[cols].astype(object).where(out[cols].notna(), None)
(
    spark.createDataFrame(out, SCHEMA).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.TBL_FW_TIMING)
)
spark.sql(f"""
    COMMENT ON TABLE {config.TBL_FW_TIMING} IS
    'Flood warning timing validation (27_fw_timing_validation). One row per upward crossing of a gauge RES threshold (kind res_event:
     outcome on time, too short, late or missed, with lead_hours against target_lead_hours) or per flood warning with no crossing
     (kind warning_only: RES not reached, or no readings). trigger_time is the FW threshold crossing before it and issue_delay_hours the
     time the warning took after it. Restricted: it is built from the thresholds.'
""")
print(f"Written {spark.table(config.TBL_FW_TIMING).count():,} rows to {config.TBL_FW_TIMING}")
dbutils.notebook.exit("success")
